# The BCJR over the air

Puts fresh MS-PRS blocks on air between the two Plutos and decodes every burst with the simulator's BCJR equaliser (MAP), with no a priori and no outer decoder, so what comes out is the error rate of the coded bits as sent. Each block is 4,998 random information bits, (5,7)-encoded and interleaved into 10,000 coded bits, MS-PRS modulated into 5,001 symbols behind a preamble, and sent at a transmit attenuation drawn per block. The receiver finds the burst and corrects one frequency offset and one phase (`ota/sync.py`); the in-phase rail goes to the BCJR.

**This transmits at 448 MHz for as long as it runs.** Stop it with the kernel's interrupt; the transmitter is silenced either way. Each run's log lands in `results/sessions/ota_bcjr_<UTC start>/`, which git ignores.

Needs `pip install pyadi-iio`, the built simulator (`build/bin/msprs`) and `bench.local.json` at the repo root with the two serials.

In [ ]:
BURSTS = 2000               # stop after this many decoded bursts
L0, FAMILY = 3, "balanced"
ATTEN_DB = (30.0, 60.0)     # transmit attenuation, drawn per block
RX_GAIN_DB = 65.0
CARRIER_HZ, FS = 448e6, 1e6

In [ ]:
import sys, csv, json, datetime, pathlib
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from ota.waveform import ROOT, Waveform
from ota.sync import bursts, matches
from ota.baseline import bcjr_llr
from ota.radio import Bench

wave = Waveform(L0=L0, family=FAMILY)
rng = np.random.default_rng()
run = ROOT / "results" / "sessions" / ("ota_bcjr_" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
run.mkdir(parents=True)
print(run)

In [ ]:
done, missed = 0, 0
with open(run / "log.csv", "w", newline="") as fh, Bench(carrier=CARRIER_HZ, fs=FS, rx_gain=RX_GAIN_DB) as bench:
    log = csv.writer(fh)
    log.writerow(["atten_db", "esn0_db", "cfo_hz", "ber"])
    try:
        while done < BURSTS:
            _, sent, symbols = wave.block(rng)
            atten = rng.uniform(*ATTEN_DB)
            bench.send(wave.samples(symbols), atten)
            got = [b for b in bursts(bench.capture(), wave, FS) if matches(b, symbols)]
            if not got:            # nothing above the detection floor, or only the old burst
                missed += 1
                continue
            for b in got:
                llr = bcjr_llr(b.symbols.real, b.sigma, wave.N, L0, FAMILY)
                log.writerow([atten, b.esn0_db, b.cfo_hz, np.mean((llr < 0) != sent)])
            done += len(got)
            print(f"\r{done} bursts, {missed} captures missed", end="")
    except KeyboardInterrupt:
        print("\nstopped")
print(f"\ntransmitter silenced; {done} bursts in {run / 'log.csv'}")

Error rate against the $E_s/N_0$ measured on the preamble, in 1 dB bins, next to the simulator's uncoded curve on the Gaussian channel. The coded bits ride two per symbol, so that record's $E_b/N_0$ sits 3.01 dB below $E_s/N_0$. Bins with fewer than five bursts or no errors are left out. Set `run` to an earlier session to redraw it without the radios.

In [ ]:
log = np.genfromtxt(run / "log.csv", delimiter=",", names=True)
edges = np.arange(np.floor(log["esn0_db"].min()), log["esn0_db"].max() + 1, 1.0)
k = np.digitize(log["esn0_db"], edges) - 1
n = np.bincount(k, minlength=len(edges))[:len(edges) - 1]
ber = np.bincount(k, log["ber"], minlength=len(edges))[:len(edges) - 1] / np.maximum(n, 1)
mid = (edges[:-1] + edges[1:]) / 2
shown = (n >= 5) & (ber > 0)

sim = sorted((json.loads(f.read_text()) for f in (ROOT / "results" / "ber" / f"nsm_L{L0}_{FAMILY}_uncoded").glob("snr_*dB.json")),
             key=lambda r: r["eb_no_db"])

fig, ax = plt.subplots(figsize=(4.5, 3.2))
ax.semilogy([r["eb_no_db"] + 10 * np.log10(2) for r in sim], [r["ber"] for r in sim],
            color="#898781", lw=1.5, label="simulator, Gaussian channel")
ax.semilogy(mid[shown], ber[shown], "o", color="#0b0b0b", ms=4, label="over the air")
ax.set(xlabel="$E_s/N_0$ from the preamble (dB)", ylabel="coded-bit error rate, BCJR (MAP)")
ax.grid(True, color="#e1e0d9", lw=0.6)
ax.legend(frameon=False)
fig.savefig(run / "ber.png", dpi=200, bbox_inches="tight")